# ConvertThis module processes our bronze documents. It is the most involved and time consuming of the modules. We leverage the Docling framework to abstract away the layout analysis of a document.In order to make this useful downstream for multimodal vector search, we need three things:- Exported tables, images, and pages- A reloadable and cachable conversion- Vector search ready text chunks that also incorporate tables and figuresThis notebook is designed to be used with a classic cluster using ML Runtime 15.4 LTS, with CPU compute. It was tested with a STANDARD_e8_v3 with 6 workers in Azure. The goal is to reduce cost as much as possible by using cheap CPU workers with high utilization across workers.

In [0]:
%pip install uv

In [0]:
%sh uv pip install '.[convert]'

In [0]:
%restart_python

In [0]:
from pathlib import Pathpdf_input_path = Path("tests/data/pid_diagram.pdf")output_dir = Path("data/processed")  # Define output directory

In [0]:
from databricks.sdk import WorkspaceClientfrom openai import OpenAIfrom src.utils import get_tokenw = WorkspaceClient()workspace_url = w.config.hosttoken = get_token(w)llm_model = "databricks-claude-3-7-sonnet"llm_client = OpenAI(    api_key=token,    base_url=f"{workspace_url}/serving-endpoints",)

In [0]:
import warningswarnings.filterwarnings(    "ignore",    message="'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.",)

## Ray MAUD Docling ConverterThe MAUD acclerator extends Docling in order to process the tables, pages, and figures as well as the document hierarchy. We setup the whole converter pipeline within a single object

In [0]:
from docling.document_converter import DocumentConverterconverter = DocumentConverter()doc = converter.convert(source=pdf_input_path).document

In [0]:
doc.model_dump()

In [0]:
from docling.datamodel.base_models import InputFormatfrom docling.document_converter import PdfFormatOptionfrom src.document.converters import MAUDPipelineOptions, MAUDConverter, MAUDPipelineimport pandas as pdmaud_options = MAUDPipelineOptions(    llm_client=llm_client,    llm_model="databricks-claude-3-7-sonnet",    max_tokens=200,    clf_client=None,    clf_model="dummy_clf",    do_picture_description=True,    do_page_description=True,    generate_page_images=True,    generate_picture_images=True,    generate_table_images=True,)

In [0]:
converter = MAUDConverter(    input_path=pdf_input_path,    output_dir=output_dir,    llm_client=maud_options.llm_client,    llm_model=maud_options.llm_model,    max_tokens=maud_options.max_tokens,    overwrite=True,    format_options={        InputFormat.PDF: PdfFormatOption(            pipeline_cls=MAUDPipeline,            pipeline_options=maud_options,        )    },)document = converter.convert()

In [0]:
document.model_dump().keys()

In [0]:
converter.save_document()chunks = converter.chunk()

In [0]:
pd.DataFrame(chunks)